## Connection Establishment & HTML Payload Retrieval

**Purpose:** Establish a stable HTTP connection to the target website and retrieve the raw HTML document for processing without triggering automated bot defenses.

**Role of the Code:**
* **Network Execution**: Initiates the web call to fetch the site's contents.
* **Header Injection (The Disguise)**: Many websites automatically block scripts. By sending a specific `User-Agent` header, the script mimics a normal human using a Windows computer and a Chrome web browser, successfully bypassing basic bot-blocking security.
* **Validation**: Checks that the status code equals **200 (OK)**. A successful 200 code clears the script to proceed, preventing crashes from dead links (like a 404 error).

**Output:**
When executed, this block outputs `Status Code: 200` followed by the `--- HTML Preview ---`. Because the entire underlying HTML code is massive and messy, the script deliberately slices the text to print only the first 500 characters. This output is crucial as it proves network connectivity is stable and the raw HTML blueprint has been successfully downloaded into local memory.

In [ ]:
import requests

url = "https://fuelprice.ph/"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win    64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
}

print("Fetching data...")
response = requests.get(url, headers=headers)
print(f"Status Code: {response.status_code}")

if response.status_code == 200:
    print("\n--- HTML Preview ---")
    print(response.text[:500])
else:
    print("Request failed.")

Fetching data...
Status Code: 200

--- HTML Preview ---
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">

  
  <title>Philippines Fuel Prices Today — Jun 2, 2026 | fuelprice.ph</title>
  <meta name="description" content="Philippines fuel prices updated every Tuesday. Compare diesel, diesel plus, gasoline (Unleaded 91, Premium 95), LPG, and kerosene prices from Shell, Petron, Caltex, Seaoil, Phoenix, and Cleanfuel. Trip cost calculator and weekly price updates.">



## Payload Interception & Macro Data Extraction

**Purpose:** Extract global macro-economic indicators (ticker) and granular retail fuel prices using an ELT (Extract, Load, Transform) approach, relying on Python's Regular Expression (`re`) module instead of fragile HTML parsers.

**Role of the Code:**
* **Regex Anchoring**: Targets specific HTML classes (`ticker-highlight`) to extract numbers, bypassing false positives in the site's text paragraphs. It uses flags to force Python to jump over invisible line breaks hidden in the raw HTML.
* **The "Hidden API" Interception**: Network analysis revealed the developers embedded the database response directly into a hidden `<script>` tag. The regex pattern scans the static HTML to isolate the exact text block containing this JSON dictionary.
* **Data Parsing**: Instantly converts the intercepted string payload into a navigable, multi-level Python dictionary.
* **Structuring & Enrichment**: Iterates through the target fuel keys (unleaded-91, premium-95, diesel, diesel-plus) and flattens the nested data into a single array. It injects system execution metadata (timestamps) and macro context (`dubai_crude`, `usd_php_rate`) directly into every single retail row.

**Output:**
The output `Success: Intercepted internal JSON payload.` validates the structural pivot of the pipeline. It proves the regex engine successfully bypassed the empty HTML tables and located the hidden JavaScript variable. The subsequent message `Extraction Complete: 44 enriched records staged...` indicates the payload was successfully unpacked, flattened, and merged with the live Dubai Crude and USD/PHP macro data without throwing any parsing errors.

In [2]:
import re
import json
import datetime

# Define the exact fuel types targeted for the routing application
target_fuels = ['unleaded-91', 'premium-95', 'diesel', 'diesel-plus']

# Map URL slugs to clean display names for the analytics layer
fuel_display_map = {
    'unleaded-91': 'Unleaded 91',
    'premium-95': 'Premium 95',
    'diesel': 'Diesel',
    'diesel-plus': 'Diesel Plus'
}

# Initialize the array to hold fully structured records
raw_brand_data = []

# --- EXTRACT LIVE TICKER MACRO DATA (HTML ANCHORED) ---
# We use single quotes ' ' on the outside so we can use double quotes " " for the HTML class inside
crude_pattern = r'DUBAI CRUDE.*?\$(\d+\.\d+)'
forex_pattern = r'USD/PHP.*?class="ticker-highlight">\s*(\d+\.\d+)'

# We add '| re.DOTALL' to force Python to jump over the invisible line breaks
crude_match = re.search(crude_pattern, response.text, re.IGNORECASE | re.DOTALL)
forex_match = re.search(forex_pattern, response.text, re.IGNORECASE | re.DOTALL)

dubai_crude = float(crude_match.group(1)) if crude_match else None
usd_php_rate = float(forex_match.group(1)) if forex_match else None
# --------------------------------------

# Scan the raw HTML response text for the target JavaScript variable
pattern = r"const brandData\s*=\s*({.*?});"
match = re.search(pattern, response.text)

if match:
    # Extract and parse the embedded JSON payload
    json_string = match.group(1)
    api_data = json.loads(json_string)
    print("Success: Intercepted internal JSON payload.")
    
    # Generate system execution metadata for lineage tracking and deduplication
    current_timestamp = datetime.datetime.now().isoformat()
    current_date_str = datetime.datetime.now().strftime("%Y-%m-%d")
    
    # Loop over target fuel categories to extract specific objects
    for fuel in target_fuels:
        if fuel in api_data:
            brands_list = api_data[fuel]
            
            for brand_info in brands_list:
                # Map source parameters and inject operational tracking dimensions
                raw_brand_data.append({
                    "ingestion_run_id": current_timestamp,
                    "extraction_timestamp": current_timestamp,
                    "week_date": current_date_str,
                    "source": "fuelprice.ph",
                    "fuel_type": fuel,
                    "fuel_display_name": fuel_display_map.get(fuel),
                    "brand": brand_info.get("n"),
                    "raw_price": brand_info.get("price"),
                    "price_unit": "PHP_per_liter",
                    "raw_change": brand_info.get("chg"),
                    "change_direction": brand_info.get("dir"), # Added direct movement string
                    "status": brand_info.get("status"),
                    "verified_date": brand_info.get("date"),
                    "dubai_crude_usd_per_barrel": dubai_crude,  # Injected macro context
                    "usd_php_rate": usd_php_rate               # Injected macro context
                })

    print(f"Extraction Complete: {len(raw_brand_data)} enriched records staged (with Ticker Macro Data).")

else:
    print("Error: Target variable 'brandData' not found in source text.")


Success: Intercepted internal JSON payload.
Extraction Complete: 44 enriched records staged (with Ticker Macro Data).


## Data Quality Assurance

**Purpose:** Perform an automated system audit to ensure the JSON extraction captures the complete dataset without dropping any records or nested objects before moving the data into physical storage.

**Role of the Code:**
* **Aggregation Dictionary**: Creates an empty dictionary to maintain a running operational tally of records based on their fuel type.
* **Validation Loop**: Iterates through the newly built array. If it encounters a fuel type for the first time, it initializes the count at 1; otherwise, it adds 1 to the existing total.
* **Checkpointing**: Prints a cleanly formatted summary. This provides an immediate visual safeguard allowing developers to verify the extraction against the live website's brand listings.

**Output:**
The terminal outputs a precise breakdown: `UNLEADED-91: 13 brands extracted`, `PREMIUM-95: 12 brands extracted`, `DIESEL: 13 brands extracted`, and `DIESEL-PLUS: 6 brands extracted`. This aggregation acts as a vital checksum. Because these individual counts mathematically sum up to exactly `44`—matching the total records reported in the extraction phase—it definitively proves the extraction loop captured the complete dataset without dropping a single record.

---

In [3]:
# Create an empty dictionary to keep a running tally
fuel_counts = {}

# Loop through every single record we just extracted
for record in raw_brand_data:
    fuel = record["fuel_type"]
    
    # If we haven't seen this fuel type yet, start its count at 1
    if fuel not in fuel_counts:
        fuel_counts[fuel] = 1
    # If we have seen it, just add 1 to its running total
    else:
        fuel_counts[fuel] += 1

print("--- Data Verification: Brand Count by Fuel Type ---")

# Print out the final tally nicely
for fuel, count in fuel_counts.items():
    print(f"{fuel.upper()}: {count} brands extracted")

print(f"\nTotal Records Extracted: {len(raw_brand_data)}")

--- Data Verification: Brand Count by Fuel Type ---
UNLEADED-91: 13 brands extracted
PREMIUM-95: 12 brands extracted
DIESEL: 13 brands extracted
DIESEL-PLUS: 6 brands extracted

Total Records Extracted: 44


## Local Storage Partitioning & JSON Serialization

**Purpose:** Securely write the processed data from the computer's temporary memory to a permanent file on the physical disk, mimicking the behavior of an automated drop into a cloud data lake (e.g., AWS S3).

**Role of the Code:**
* **Safe Directory Management**: Defines the target folder path and safely creates it. Flags are used to prevent the script from crashing with a "File Exists" error on subsequent runs.
* **Dynamic Time-Series Partitioning**: Appends the current execution date to the filename (e.g., `raw_fuel_prices_2026-06-20.json`). This partitions the data by day, creating a clean historical record without overwriting past runs.
* **Secure Serialization**: Opens the file using standard `utf-8` encoding. Crucially, it disables ASCII enforcement to protect special symbols (like the `₱` and trend arrows like `↓`) from becoming corrupted text, while formatting the file to be easily readable by humans during debugging.

**Output:**
The output `Export Success: Staging lifecycle complete.` followed by the `Target location: data/raw/raw_fuel_prices_YYYY-MM-DD.json` signifies the end of the ingestion lifecycle. It confirms that the Python script successfully opened the local file system, applied the correct encoding configurations to preserve international symbols, and effectively committed the JSON array to disk, ensuring the data is preserved for downstream transformation processes.

In [4]:
import os
import json

# Ensure local storage pathing exists
output_dir = "data/raw"
os.makedirs(output_dir, exist_ok=True)

# Build dynamic, date-partitioned filename
filename = f"raw_fuel_prices_{current_date_str}.json"
filepath = os.path.join(output_dir, filename)

# Write the complete dictionary array ensuring symbol safety
with open(filepath, 'w', encoding='utf-8') as file:
    json.dump(raw_brand_data, file, ensure_ascii=False, indent=4)
print("Export Success: Staging lifecycle complete.")
print(f"Target location: {filepath}")

Export Success: Staging lifecycle complete.
Target location: data/raw\raw_fuel_prices_2026-06-22.json
